In [33]:
from google import genai
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
import json
import duckdb
import pandas as pd

In [8]:
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMENI_API_KEY"))

response = client.models.generate_content(
    model = "gemini-2.5-flash", contents="Tell me a programming joke"
)
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Why do programmers prefer dark mode?

Because bugs are attracted to light!"""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='uZm-aMmbJae0kdUPx8OH2A4',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=15,
    prompt_token_count=6,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=6
      ),
    ],
    thoughts_token_count=1114,
    total_token_count=1135
  )
)

In [ ]:
print(response.text)

In [ ]:
def ask_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

ask_llm("Ge mig ett Göteborgs skämt")

In [10]:
response = ask_llm("""
    Du är en expert inom köp och sälj av bostäder, likt en proffsig mäklare.
    Generera bostadspriser, månadsavgifter, address, stad, boarea i jsonformat (ej markdown)

    Exempel:
            {
                "address": "Fågelvägen 5,
                "price_sek": 3000000,
                "city": "Göteborg",
                "monthly_fee": 4000,
                "area": 60
            }   
                   
    Ge mig en lista på 5 bostäder
""")

In [ ]:
print(response)

In [24]:
class Apartment(BaseModel):
    address: str
    city: str
    price_sek: int = Field(gt=1000000, lt=9000000)
    monthly_fee: int
    area: int

class ApartmentList(BaseModel):
    objects: list[Apartment]

apartments = ApartmentList.model_validate({"objects": json.loads(response)})
apartments

ApartmentList(objects=[Apartment(address='Karlavägen 10A', city='Stockholm', price_sek=7500000, monthly_fee=4500, area=85), Apartment(address='Linnégatan 25', city='Göteborg', price_sek=4800000, monthly_fee=3800, area=70), Apartment(address='Davidshallsgatan 8', city='Malmö', price_sek=3200000, monthly_fee=3500, area=65), Apartment(address='Svartbäcksgatan 15A', city='Uppsala', price_sek=2750000, monthly_fee=2900, area=50), Apartment(address='Tegnérgatan 30', city='Stockholm', price_sek=5200000, monthly_fee=3200, area=45)])

In [25]:
apartments.objects

[Apartment(address='Karlavägen 10A', city='Stockholm', price_sek=7500000, monthly_fee=4500, area=85),
 Apartment(address='Linnégatan 25', city='Göteborg', price_sek=4800000, monthly_fee=3800, area=70),
 Apartment(address='Davidshallsgatan 8', city='Malmö', price_sek=3200000, monthly_fee=3500, area=65),
 Apartment(address='Svartbäcksgatan 15A', city='Uppsala', price_sek=2750000, monthly_fee=2900, area=50),
 Apartment(address='Tegnérgatan 30', city='Stockholm', price_sek=5200000, monthly_fee=3200, area=45)]

In [ ]:
apartments.objects[1].address, apartments.objects[1].city

In [ ]:
addresses = [apartment.address for apartment in apartments.objects]
addresses

In [ ]:
addresses = [
    apartment.address
    for apartment in apartments.objects
    if apartment.price_sek < 4000000
]
addresses

Get address, city, price, monthly_fee for the interval 4M - 8M

In [ ]:
df_staging_apt = [
    apartment.model_dump(include={"address", "city", "price_sek", "monthly_fee"})
    for apartment in apartments.objects
]

In [ ]:
# model dump to export the data to a dict, json or df
address_interval = pd.DataFrame(
    [
        apartment.model_dump(include={"address", "city", "price_sek", "monthly_fee"})
        for apartment in apartments.objects
        if 4_000_000 <= apartment.price_sek <= 8_000_000
    ])
address_interval

In [ ]:
con = duckdb.connect("apartments.duckdb")
con.execute("CREATE TABLE IF NOT EXISTS apartments AS SELECT * FROM df")
con.execute("INSERT INTO apartments SELECT * FROM df")
con.close()

In [ ]:
apartments.model_dump()

In [46]:
apartments.model_dump_json()

'{"objects":[{"address":"Karlavägen 10A","city":"Stockholm","price_sek":7500000,"monthly_fee":4500,"area":85},{"address":"Linnégatan 25","city":"Göteborg","price_sek":4800000,"monthly_fee":3800,"area":70},{"address":"Davidshallsgatan 8","city":"Malmö","price_sek":3200000,"monthly_fee":3500,"area":65},{"address":"Svartbäcksgatan 15A","city":"Uppsala","price_sek":2750000,"monthly_fee":2900,"area":50},{"address":"Tegnérgatan 30","city":"Stockholm","price_sek":5200000,"monthly_fee":3200,"area":45}]}'

In [47]:
with open("apartments.json", "w") as json_file:
    json_file.write(apartments.model_dump_json(indent=3))